# Curry College SMIF – Analytics Notebook
**How to use this page:** each grey box is a *cell*. Click the ▶ Play button on the left of a cell (or press Shift+Enter) and wait for it to finish before moving to the next one. Run them **in order, top to bottom**.

You never need to type code. The only cells you may edit are the ones marked ✏️ **EDIT ME**.

*If you see "Runtime disconnected", just choose Runtime → Run all.*

## Step 1 – Set up (run once, ~20 seconds)

In [ ]:
!pip -q install yfinance openpyxl

## Step 2 – Load the SMIF toolkit 🔒 (do not edit)
This cell saves our analysis code. It prints nothing; that's normal.

In [ ]:
%%writefile smif_models.py
"""
Curry College SMIF - reusable analytics toolkit
------------------------------------------------
Usage (from a terminal, in this folder):
    pip install pandas numpy scipy openpyxl yfinance
    python smif_models.py update                 # download/refresh daily prices -> data/prices.csv
    python smif_models.py stats                  # return, vol, Sharpe, beta, CAPM, alpha, drawdown
    python smif_models.py optimize               # max-Sharpe, min-variance, risk-parity (+ your current weights)
    python smif_models.py candidates             # marginal impact of each new ETF on the current portfolio
    python smif_models.py factors                # factor regressions (ETF-based proxy factors)
    python smif_models.py holdings FILE.xlsx     # top-10 vs not-top-10 attribution from SMIF Portfolio sheet
    python smif_models.py navcheck               # price vs. NAV (premium/discount), best effort
    python smif_models.py all
Edit the CONFIG block below each semester; nothing else needs to change.
"""
import sys, os, warnings
import numpy as np, pandas as pd
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ------------------------------ CONFIG ---------------------------------------
CURRENT = {"XLE": 31*62.21, "XLI": 3*168.78, "VCR": 3*403.41,
           "VHT": 9*346.70, "VGT": 4*111.84, "VOO": 4*691.89}   # $ value -> weights (update after each trade)
CANDIDATES = {"XLU": "Utilities", "IBB": "Biotech", "PAVE": "Infrastructure",
              "XLP": "Consumer Staples", "SMH": "Semiconductors",
              "IJH": "Mid-cap", "IYG": "Financial Services"}
MARKET = "VTI"                     # market proxy for beta/CAPM (matches the class Excel model)
START = "2021-07-01"
LOOKBACK_YEARS = None              # None = full sample; or e.g. 3
MAX_W, MIN_W = 0.35, 0.0           # per-ETF caps for the optimizers (long-only, no leverage)
RF_FALLBACK = 0.04                 # annual risk-free if FRED download is unavailable
ADD_SIZE = 0.10                    # candidate test: add 10% of portfolio, funded pro-rata
# ETF-based factor proxies (long-short built from ETFs so it works with no outside data)
FACTOR_ETFS = ["MTUM", "QUAL", "USMV", "VLUE", "IWM", "VTV", "VUG"]
DATA = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")
os.makedirs(DATA, exist_ok=True)
PRICES = os.path.join(DATA, "prices.csv")
# -----------------------------------------------------------------------------
TD = 252


def tickers_all():
    return sorted(set(CURRENT) | set(CANDIDATES) | {MARKET} | set(FACTOR_ETFS))


def update():
    """Pull adjusted closes (total-return basis) + 3M T-bill; merge into data/prices.csv. Re-run any time."""
    import yfinance as yf
    px = yf.download(tickers_all(), start=START, auto_adjust=True, progress=False)["Close"]
    px.index = pd.to_datetime(px.index).tz_localize(None)
    try:
        rf = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO",
                         index_col=0, parse_dates=True).iloc[:, 0].replace(".", np.nan).astype(float) / 100
        px["RF"] = rf.reindex(px.index).ffill()
    except Exception as e:
        print("FRED unavailable, using fallback rf:", e)
        px["RF"] = RF_FALLBACK
    px.to_csv(PRICES)
    print(f"saved {px.shape} -> {PRICES}  (last date {px.index[-1].date()})")


def load(seed_xlsx=None):
    """Load prices. If no csv yet, seed from the class Excel file (6 ETFs + VTI + RF)."""
    if os.path.exists(PRICES):
        px = pd.read_csv(PRICES, index_col=0, parse_dates=True)
    elif seed_xlsx:
        raw = pd.read_excel(seed_xlsx, sheet_name="Trading Data Original", header=None, skiprows=3)
        px = raw.iloc[:, 1:9].copy()
        px.columns = ["Date", "XLE", "XLI", "VGT", "VHT", "VCR", "VOO", "VTI"]
        px = px.dropna(subset=["Date"]).set_index("Date").astype(float)
        px["RF"] = raw.iloc[:, 27].values[:len(px)]
        px.index = pd.to_datetime(px.index).normalize()
    else:
        sys.exit("No data/prices.csv - run:  python smif_models.py update")
    px = px.sort_index()
    if LOOKBACK_YEARS:
        px = px[px.index >= px.index[-1] - pd.DateOffset(years=LOOKBACK_YEARS)]
    return px


def split(px):
    rf_daily = px["RF"].ffill().fillna(RF_FALLBACK) / TD
    assets = px.drop(columns="RF")
    rets = np.log(assets / assets.shift(1))
    return rets, rf_daily.reindex(rets.index).ffill()


def cur_weights(cols):
    w = pd.Series(CURRENT).reindex(cols).fillna(0)
    return w / w.sum()


# ------------------------------ 1. STATS / CAPM -------------------------------
def stats(px):
    r, rf = split(px)
    ex = r.sub(rf, axis=0)
    m = ex[MARKET]
    rows = {}
    for c in r.columns:
        d = pd.concat([ex[c], m], axis=1).dropna()
        if len(d) < 60:
            continue
        beta = np.cov(d.iloc[:, 0], d.iloc[:, 1])[0, 1] / d.iloc[:, 1].var()
        ann_r = r[c].dropna().mean() * TD
        vol = r[c].dropna().std() * np.sqrt(TD)
        rfa = rf.mean() * TD
        capm = rfa + beta * (m.mean() * TD)
        cum = np.exp(r[c].dropna().cumsum())
        dd = (cum / cum.cummax() - 1).min()
        # alpha t-stat from regression
        X = np.column_stack([np.ones(len(d)), d.iloc[:, 1]])
        b, *_ = np.linalg.lstsq(X, d.iloc[:, 0].values, rcond=None)
        res = d.iloc[:, 0].values - X @ b
        se = np.sqrt(res.var(ddof=2) * np.linalg.inv(X.T @ X)[0, 0])
        rows[c] = dict(ann_return=ann_r, vol=vol, sharpe=(ann_r - rfa) / vol, beta=beta,
                       capm_return=capm, alpha=ann_r - capm, alpha_tstat=b[0] / se, max_drawdown=dd)
    out = pd.DataFrame(rows).T
    print("\n=== Risk / return / CAPM (log returns, annualised; alpha t-stat |t|>2 is 'real') ===")
    print(out.round(3).to_string())
    return out


# ------------------------------ 2. OPTIMIZERS ---------------------------------
def _port(w, mu, cov, rfa):
    r = w @ mu; v = np.sqrt(w @ cov @ w)
    return r, v, (r - rfa) / v


def optimize(px, universe=None):
    r, rf = split(px)
    cols = universe or [c for c in CURRENT if c in r.columns]
    R = r[cols].dropna()
    mu, cov, rfa = R.mean().values * TD, R.cov().values * TD, rf.reindex(R.index).mean() * TD
    n = len(cols)
    bnds = [(MIN_W, MAX_W)] * n
    cons = {"type": "eq", "fun": lambda w: w.sum() - 1}
    x0 = np.ones(n) / n
    sols = {
        "max_sharpe": minimize(lambda w: -_port(w, mu, cov, rfa)[2], x0, bounds=bnds, constraints=cons).x,
        "min_variance": minimize(lambda w: w @ cov @ w, x0, bounds=bnds, constraints=cons).x,
        "risk_parity": minimize(lambda w: np.sum((w * (cov @ w) / (w @ cov @ w) - 1/n) ** 2),
                                x0, bounds=[(1e-4, 1)] * n, constraints=cons).x,
        "equal_weight": x0,
        "CURRENT": cur_weights(cols).values,
    }
    tab = pd.DataFrame(sols, index=cols).T
    perf = pd.DataFrame({k: _port(v, mu, cov, rfa) for k, v in sols.items()},
                        index=["exp_ret", "vol", "sharpe"]).T
    print(f"\n=== Optimal weights (cap {MAX_W:.0%}/ETF, long-only, {R.index[0].date()}→{R.index[-1].date()}) ===")
    print((tab * 100).round(1).to_string())
    print("\nIn-sample performance (these are IN-SAMPLE; see walk-forward below)")
    print(perf.round(3).to_string())
    walk_forward(R, rf, bnds, cons)
    return tab


def walk_forward(R, rf, bnds, cons, train_yrs=2, step_m=3):
    """Out-of-sample test: fit weights on trailing window, hold for next quarter. Sharpe max usually
    looks great in-sample and mediocre here - THE key teaching point."""
    n = R.shape[1]; x0 = np.ones(n) / n
    dates = R.index
    starts = pd.date_range(dates[0] + pd.DateOffset(years=train_yrs), dates[-1], freq=f"{step_m}MS")
    out = {k: [] for k in ["max_sharpe", "min_variance", "equal_weight"]}
    for s in starts:
        tr = R[(R.index < s) & (R.index >= s - pd.DateOffset(years=train_yrs))]
        te = R[(R.index >= s) & (R.index < s + pd.DateOffset(months=step_m))]
        if len(te) < 5:
            continue
        mu, cov = tr.mean().values * TD, tr.cov().values * TD
        rfa = rf.reindex(tr.index).mean() * TD
        ws = {"max_sharpe": minimize(lambda w: -_port(w, mu, cov, rfa)[2], x0, bounds=bnds, constraints=cons).x,
              "min_variance": minimize(lambda w: w @ cov @ w, x0, bounds=bnds, constraints=cons).x,
              "equal_weight": x0}
        for k, w in ws.items():
            out[k].append(te @ w)
    res = {}
    for k, v in out.items():
        if v:
            s = pd.concat(v)
            res[k] = dict(oos_ann_ret=s.mean() * TD, oos_vol=s.std() * np.sqrt(TD),
                          oos_sharpe=(s.mean() * TD - rf.mean() * TD) / (s.std() * np.sqrt(TD)))
    print(f"\n=== Walk-forward out-of-sample ({train_yrs}y train, rebalance every {step_m}m) ===")
    print(pd.DataFrame(res).T.round(3).to_string())


# ------------------------------ 3. CANDIDATE SCREEN ---------------------------
def candidates(px):
    r, rf = split(px)
    cur = [c for c in CURRENT if c in r.columns]
    w0 = cur_weights(cur).values
    rows = {}
    for c in CANDIDATES:
        if c not in r.columns:
            continue
        D = r[cur + [c]].dropna()
        if len(D) < 120:
            print(f"skip {c}: only {len(D)} days"); continue
        mu, cov = D.mean().values * TD, D.cov().values * TD
        rfa = rf.reindex(D.index).mean() * TD
        base = np.append(w0, 0)
        new = np.append(w0 * (1 - ADD_SIZE), ADD_SIZE)
        b, n_ = _port(base, mu, cov, rfa), _port(new, mu, cov, rfa)
        port_r = D[cur] @ w0
        corr_p = np.corrcoef(port_r, D[c])[0, 1]
        corr_cur = D[cur].corrwith(D[c])
        # diversification: R^2 of candidate explained by current holdings (low = unique exposure)
        X = np.column_stack([np.ones(len(D)), D[cur].values])
        beta, *_ = np.linalg.lstsq(X, D[c].values, rcond=None)
        r2 = 1 - ((D[c].values - X @ beta).var() / D[c].var())
        rows[c] = {"sector": CANDIDATES[c], "days": len(D), "corr_w_portfolio": corr_p,
                   "most_similar": corr_cur.idxmax(), "max_corr": corr_cur.max(),
                   "R2_by_holdings": r2, "d_vol": n_[1] - b[1], "d_exp_ret": n_[0] - b[0],
                   "d_sharpe": n_[2] - b[2]}
    out = pd.DataFrame(rows).T.sort_values("d_sharpe", ascending=False)
    print(f"\n=== Marginal impact of adding {ADD_SIZE:.0%} of each candidate (funded pro-rata) ===")
    print("Low R2_by_holdings / low corr = more new diversification. d_sharpe uses historical means (noisy!); "
          "d_vol is the more reliable column.")
    print(out.to_string(float_format=lambda x: f"{x:.3f}"))
    return out


# ------------------------------ 4. FACTORS ------------------------------------
def factors(px):
    """Regress each ETF on market + size + value + momentum + quality + low-vol, built from ETF spreads.
    Swap in Ken French data (mba.tuck.dartmouth.edu/pages/faculty/ken.french) if you want textbook factors."""
    r, rf = split(px)
    F = pd.DataFrame(index=r.index)
    need = lambda *x: all(k in r.columns for k in x)
    if need(MARKET):         F["MKT"] = r[MARKET] - rf
    if need("IWM", "VOO"):   F["SIZE"] = r["IWM"] - r["VOO"]          # small minus large
    if need("VTV", "VUG"):   F["VALUE"] = r["VTV"] - r["VUG"]         # value minus growth
    if need("MTUM", MARKET): F["MOM"] = r["MTUM"] - r[MARKET]         # momentum vs market
    if need("QUAL", MARKET): F["QUALITY"] = r["QUAL"] - r[MARKET]
    if need("USMV", MARKET): F["LOWVOL"] = r["USMV"] - r[MARKET]
    F = F.dropna(how="all")
    targets = [c for c in list(CURRENT) + list(CANDIDATES) if c in r.columns]
    rows = {}
    for c in targets:
        D = pd.concat([r[c] - rf, F], axis=1).dropna()
        if len(D) < 120: continue
        X = np.column_stack([np.ones(len(D)), D.iloc[:, 1:].values])
        y = D.iloc[:, 0].values
        b, *_ = np.linalg.lstsq(X, y, rcond=None)
        res = y - X @ b
        se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * res.var(ddof=X.shape[1]))
        rows[c] = {"alpha_ann": b[0] * TD, "alpha_t": b[0] / se[0], "R2": 1 - res.var() / y.var(),
                   **{f"b_{n}": v for n, v in zip(F.columns, b[1:])},
                   **{f"t_{n}": v / s for n, v, s in zip(F.columns, b[1:], se[1:])}}
    out = pd.DataFrame(rows).T
    print("\n=== Multi-factor exposure (b_ = loading, t_ = t-stat) ===")
    print(out.round(2).to_string())
    return out


# ------------------------------ 5. TOP10 vs NOT-TOP10 -------------------------
def holdings(xlsx):
    """Read the 'portfolio performance' sheet; split return into top-10 stocks vs rest, and rank contributors."""
    raw = pd.read_excel(xlsx, header=None)
    hdr = raw.index[raw[1] == "Ticker"][0]
    df = raw.iloc[hdr + 1:, 1:11].copy()
    df.columns = ["ticker", "name", "qty", "w1", "w0", "ret", "v1", "v0", "p1", "p0"]
    df = df[df["ticker"].notna() & ~df["ticker"].astype(str).str.startswith("Portfolio Holdings")]
    for c in df.columns[2:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    etfs = df[df.ticker.isin(CURRENT)]
    stocks = df[~df.ticker.isin(CURRENT) & ~df.ticker.astype(str).str.startswith("Portfolio") & df.v0.notna()].copy()
    tot0 = etfs.v0.sum()
    stocks["pnl"] = stocks.v1 - stocks.v0
    stocks["contrib_bp"] = stocks.pnl / tot0 * 1e4
    print("\n=== Stock-level contribution (look-through; basis points of total portfolio) ===")
    print(stocks.sort_values("contrib_bp")[["ticker", "w0", "ret", "pnl", "contrib_bp"]].round(2).head(8).to_string(index=False))
    print("   ...")
    print(stocks.sort_values("contrib_bp")[["ticker", "w0", "ret", "pnl", "contrib_bp"]].round(2).tail(8).to_string(index=False))
    t10_v0, t10_v1 = stocks.v0.sum(), stocks.v1.sum()
    rest_v0, rest_v1 = tot0 - t10_v0, etfs.v1.sum() - t10_v1
    print(f"\nTop-10 look-through basket: {t10_v0/tot0:.1%} of portfolio, return {t10_v1/t10_v0-1:+.2%}")
    print(f"Everything else (remaining ~{1-t10_v0/tot0:.0%}):  return {rest_v1/rest_v0-1:+.2%}")
    print(f"Whole portfolio: {etfs.v1.sum()/tot0-1:+.2%}")
    # Brinson-style: where did the 'rest' alpha come from? contribution per ETF
    etfs = etfs.assign(pnl=etfs.v1 - etfs.v0)
    etfs["contrib_bp"] = etfs.pnl / tot0 * 1e4
    print("\nETF-level contribution (bp):")
    print(etfs[["ticker", "w0", "ret", "pnl", "contrib_bp"]].round(2).to_string(index=False))
    return stocks


# ------------------------------ 6. NAV vs PRICE -------------------------------
def navcheck():
    """Premium/discount = market price / NAV - 1. yfinance exposes navPrice for many ETFs (often delayed).
    For rigor use the issuer's daily NAV CSV (SSGA, Vanguard, iShares each publish one)."""
    import yfinance as yf
    rows = {}
    for t in list(CURRENT) + list(CANDIDATES):
        try:
            i = yf.Ticker(t).info
            p, nav = i.get("regularMarketPrice") or i.get("previousClose"), i.get("navPrice")
            rows[t] = dict(price=p, nav=nav, prem_disc_pct=(p / nav - 1) * 100 if p and nav else np.nan)
        except Exception as e:
            rows[t] = dict(price=np.nan, nav=np.nan, prem_disc_pct=np.nan)
    print("\n=== Price vs NAV ===")
    print(pd.DataFrame(rows).T.round(3).to_string())


if __name__ == "__main__":
    cmd = sys.argv[1] if len(sys.argv) > 1 else "all"
    seed = os.environ.get("SMIF_SEED_XLSX")
    if cmd == "update": update(); sys.exit()
    if cmd == "holdings": holdings(sys.argv[2]); sys.exit()
    if cmd == "navcheck": navcheck(); sys.exit()
    px = load(seed)
    if cmd in ("stats", "all"): stats(px)
    if cmd in ("optimize", "all"): optimize(px)
    if cmd in ("candidates", "all"): candidates(px)
    if cmd in ("factors", "all"): factors(px)

## Step 3 – ✏️ EDIT ME: what do we own, and what are we researching?
`CURRENT` = dollars (or shares × price) we hold in each ETF. Weights are computed for you.
`CANDIDATES` = the new ETFs students are researching. To add one, copy a line and change the ticker and label.

In [ ]:
import smif_models as sm
sm.CURRENT = {"XLE": 31*62.21, "XLI": 3*168.78, "VCR": 3*403.41,
              "VHT": 9*346.70, "VGT": 4*111.84, "VOO": 4*691.89}
sm.CANDIDATES = {"XLU": "Utilities", "IBB": "Biotech", "PAVE": "Infrastructure",
                 "XLP": "Consumer Staples", "SMH": "Semiconductors",
                 "IJH": "Mid-cap", "IYG": "Financial Services"}
sm.MAX_W = 0.35        # no single ETF above 35% in the optimizer
sm.ADD_SIZE = 0.10     # candidate test: add 10% of the portfolio

## Step 4 – Download today's prices
Re-run this cell any time to refresh the data (it pulls the latest daily prices from Yahoo Finance and the 3-month T-bill rate from FRED).

In [ ]:
sm.update()
px = sm.load()
px.tail()

## Step 5 – How have our ETFs performed?
Look at the picture first, then the table.
**Questions:** Which ETF has the best Sharpe ratio? Which has a significantly positive alpha (|alpha_tstat| > 2)? Which has the biggest max drawdown?

In [ ]:
import matplotlib.pyplot as plt, numpy as np
cols = list(sm.CURRENT) + [sm.MARKET]
growth = px[cols].dropna() / px[cols].dropna().iloc[0]
growth.plot(figsize=(10,5), title="Growth of $1 (price, dividends reinvested)"); plt.show()
stats = sm.stats(px)

## Step 6 – Are our ETFs doing the same thing? (correlation)
Dark red = move together (little diversification). Lighter = more diversification.

In [ ]:
r, rf = sm.split(px)
cols = [c for c in list(sm.CURRENT)+list(sm.CANDIDATES) if c in r.columns]
corr = r[cols].corr()
fig, ax = plt.subplots(figsize=(8,6)); im = ax.imshow(corr, cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)): ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im); plt.title("Daily return correlation"); plt.show()

## Step 7 – What weights do the models suggest?
Three methods: **max Sharpe** (best return per unit of risk), **min variance** (lowest risk), **risk parity** (each ETF contributes equal risk).
**Key question:** the last table tests each method *out of sample*. Which method held up best on data it had not seen? What does that say about trusting a max-Sharpe answer?

In [ ]:
sm.optimize(px);

## Step 8 – Which new ETF adds the most diversification?
`corr_w_portfolio` and `R2_by_holdings` are low when the candidate is genuinely different from what we own. `most_similar` tells you which holding it overlaps most.
**Question:** does SMH really diversify VGT? Does XLU or XLP?

In [ ]:
sm.candidates(px);

## Step 9 – Factor exposure
What is each ETF really exposed to: the market, small vs large, value vs growth, momentum, quality, low volatility?
`b_` = exposure, `t_` = confidence (|t| > 2 is meaningful).

In [ ]:
sm.factors(px);

## Step 10 – Top-10 holdings vs the rest (optional)
Click Play, then **Choose files** and upload the *SMIF Portfolio* Excel file from your instructor.

In [ ]:
from google.colab import files
up = files.upload()
sm.holdings(list(up)[0]);

## Step 11 – Save your data
Downloads the price file so you can keep or share it.

In [ ]:
files.download('data/prices.csv')

---
### Tips
* **Nothing saves automatically.** File → Save a copy in Drive to keep your own version with your edits.
* Red error text? Re-run the cells from the top (Runtime → Run all) before asking for help.
* New to a word? `Sharpe` = excess return per unit of risk · `alpha` = return above what CAPM predicts · `beta` = sensitivity to the market · `drawdown` = worst peak-to-trough fall.